# Stage 4 — Saliency-Guided and LLM-Guided Highlight Reels

Builds short highlight reels from the full-length hypospadias repair videos,
using the frame-level phase logits and visual embeddings produced by
Stage 2's winning phase-recognition model (and, for instrument mentions in
captions, Stage 1's detector features). Three reel variants are generated
per video so that a saliency-only strategy, an LLM-only strategy, and a
combined strategy can be compared descriptively rather than assumed.

**No audio is used at any stage.** Every video this notebook touches is
first stripped of its audio track with `ffmpeg -an`; all downstream frame
reads, clip extraction, and reel assembly operate exclusively on those
audio-free copies, and the final reels carry no audio track.

**Everything that depends on which video is being processed is loaded
per-video from Stage 2's leave-one-video-out artefacts** — in particular,
saliency for a given video is always computed through the one GRU
checkpoint that held that video out during training (`load_fold_checkpoint`
in Stage 2, reimplemented identically below), never a fold that trained on
it.

## Execution order
1. Preprocessing — strip audio from every raw video
2. Load Stage 2/Stage 1 artefacts (manifest, embeddings, phase-model
   checkpoints, detector features)
3. Saliency — gradient-based frame saliency, normalisation, smoothing,
   threshold sweep, contiguous segment extraction
4. Reel 1 — saliency-only clip selection
5. Templated captioning (frame-level, phase + instruments) and sliding-window
   aggregation into clip-level captions
6. Reel 2 — LLM-only clip selection from evenly-spaced candidate clips
7. Reel 3 — combined: LLM re-ranks the saliency-shortlisted pool
8. Assembly — chronological, phase-label overlay, 2x speed, no audio,
   highest-quality encode
9. Evaluation metrics (all descriptive, no inferential testing)
10. Outputs


## 0. Colab setup


In [ ]:
# If not running in Colab this is a no-op fallback so the notebook still
# works locally against a filesystem path instead of Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — set CONFIG['project_root'] to a local path below.")


In [ ]:
%pip install -q opencv-python-headless scikit-learn scipy pandas seaborn tqdm \
    openai anthropic ollama

if IN_COLAB:
    # ffmpeg ships on the Colab image already, but this is harmless if it's
    # already present and saves a confusing failure three cells later if it
    # somehow isn't.
    !apt-get -qq install -y ffmpeg > /dev/null


In [ ]:
import os, io, json, math, time, random, re, shutil, subprocess, warnings
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from tqdm.auto import tqdm

# LLM SDKs are optional at import time — a user comparing only two of the
# three backends (or none, while developing the saliency-only Reel 1)
# shouldn't be blocked by a missing package for the third.
try:
    import openai
except ImportError:
    openai = None
try:
    import anthropic
except ImportError:
    anthropic = None
try:
    import ollama
except ImportError:
    ollama = None

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Global configuration


In [ ]:
@dataclass
class Config:
    # Stage 4 reuses Stage 2's project root as-is (same raw_videos/,
    # manifest/, features/, detector_features/, checkpoints/ — all
    # read-only from here) and only adds a stage4/ subtree beneath it for
    # this notebook's own outputs. Point this at wherever Stage 2's CONFIG
    # was pointed.
    project_root: str = "/content/drive/MyDrive/hypospadias_stage2"

    # Must match Stage 2's CFG.videos and CFG.phase_taxonomy exactly —
    # both are used to index into Stage 2's cached features, logits, and
    # checkpoint metadata, which were written against these exact values.
    videos: list = field(default_factory=lambda: [
        "1", "2", "4", "5", "6", "7", "8",
    ])
    phase_taxonomy: list = field(default_factory=lambda: [
        "background",
        "Preoperative anatomy",
        "Skin marking",
        "Epinephrine",
        "Degloving",
        "Artificial erection test",
        "Tourniquet applied",
        "Orthoplasty",
        "Glans marking",
        "Glans incision",
        "Skin mobilisation",
        "Glans wings mobilisation",
        "Urethral plate incision",
        "Tourniquet released",
        "Urethroplasty",
        "Barrier layer coverage",
        "Glansplasty",
        "Foreskin reconstruction",
        "Circumcision",
        "Skin edge closure",
        "Skin closure",
        "End of operation anatomy",
        "Dressing",
    ])
    base_fps: float = 2.0

    # Must match Stage 2's CFG.window_size / CFG.window_stride — saliency
    # is computed by re-running the exact windowing scheme the winning
    # model was trained and evaluated under, so a mismatch here would
    # silently score a model on windows it never saw.
    window_size: int = 1024
    window_stride: int = 512

    # Which Stage 2 LOOCV run to treat as "the winning phase recognition
    # model". Default is model_C (winning backbone -> GRU, visual
    # embeddings only) rather than model_D, because the saliency defined
    # below is explicitly a gradient with respect to *visual* embeddings —
    # Model D's input concatenates detector features into the same vector,
    # which would need to be masked out of the gradient anyway. Set this to
    # "model_D" only if Stage 2's Model C vs Model D comparison favours D;
    # compute_frame_saliency still restricts the gradient to the backbone
    # sub-block of the input in that case (see Section 3).
    winning_run_name: str = "model_C"

    # Fill in from Stage 2's results/backbone_sweep_summary.csv (the
    # "Winning backbone (carries forward as Model C/D input)" line printed
    # by run_backbone_sweep). Left unset on purpose, like Stage 2's
    # STAGE1_DETECTION_F1 — using the wrong backbone's feature cache would
    # silently score gibberish rather than fail loudly.
    winning_backbone: str = None

    detector_classes: list = field(default_factory=lambda: [
        "hand", "needle_driver", "forceps",
    ])

    seed: int = 42

    # --- Saliency ---
    saliency_smooth_window_s: float = 3.0
    saliency_thresholds: list = field(default_factory=lambda: [0.0, 0.1, 0.2, 0.3, 0.4])
    # Operating threshold used to build the actual candidate pools for
    # Reels 1 and 3, chosen by inspecting the sweep over
    # saliency_thresholds (Section 3.3). 0.2 is a reasonable starting
    # point (top ~80% of the normalised range excluded) — revisit after
    # looking at segment-count/duration output per video.
    saliency_operating_threshold: float = 0.2
    min_segment_s: float = 2.0  # drop sub-2s segments after thresholding as noise

    # --- Reel duration target: "2-5 minutes with 30 second leeway" is
    # read as the *final, played-back* duration a viewer experiences —
    # i.e. after the 2x speed-up applied at assembly — so the raw footage
    # budget used during clip selection is double this range.
    reel_target_min_s: float = 120.0
    reel_target_max_s: float = 300.0
    reel_leeway_s: float = 30.0
    playback_speed: float = 2.0

    # --- Captioning (Reel 2 candidate generation + Reel 2/3 clip captions) ---
    # Both sizes are run and compared (Section 5.3); caption_window_primary_s
    # is the one actually used to build Reel 2's candidate pool and the
    # reels that get assembled and evaluated end-to-end.
    caption_window_s_list: list = field(default_factory=lambda: [512.0, 32.0])
    caption_window_primary_s: float = 32.0
    caption_window_overlap: float = 0.5

    # --- LLM backends ---
    llm_backends: list = field(default_factory=lambda: ["chatgpt", "claude", "ollama"])
    openai_model: str = "gpt-4o-mini"
    anthropic_model: str = "claude-sonnet-5"
    ollama_model: str = "llama3.1"
    ollama_host: str = "http://localhost:11434"  # Ollama's own documented default
    llm_max_retries: int = 3

    # --- Assembly ---
    video_crf: int = 0  # 0 = lossless x264; raise (e.g. 15-18) if file size is prohibitive
    video_preset: str = "medium"
    overlay_font_size: int = 28


CFG = Config()

ROOT = Path(CFG.project_root)
DIRS = {
    # Read-only Stage 1/2 artefacts.
    "videos": ROOT / "raw_videos",
    "manifest": ROOT / "manifest",
    "features": ROOT / "features",
    "detector": ROOT / "detector_features",
    "checkpoints": ROOT / "checkpoints",
    # Stage 4's own outputs, kept in a dedicated subtree so nothing here
    # can collide with or overwrite a Stage 2 artefact.
    "no_audio_videos": ROOT / "stage4" / "no_audio_videos",
    "saliency": ROOT / "stage4" / "saliency",
    "candidates": ROOT / "stage4" / "candidates",
    "captions": ROOT / "stage4" / "captions",
    "selections": ROOT / "stage4" / "selections",
    "reels": ROOT / "stage4" / "reels",
    "metrics": ROOT / "stage4" / "metrics",
    "results": ROOT / "stage4" / "results",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

FOLD_CKPT_DIR = DIRS["checkpoints"] / "gru_folds"

PHASE_TO_IDX = {p: i for i, p in enumerate(CFG.phase_taxonomy)}
IDX_TO_PHASE = {i: p for p, i in PHASE_TO_IDX.items()}
N_CLASSES = len(CFG.phase_taxonomy)

# Raw-footage duration budget implied by the final 2-5-minute (+/-30s)
# target once the 2x assembly speed-up is accounted for.
RAW_DURATION_RANGE = (
    (CFG.reel_target_min_s - CFG.reel_leeway_s) * CFG.playback_speed,
    (CFG.reel_target_max_s + CFG.reel_leeway_s) * CFG.playback_speed,
)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)
if CFG.winning_backbone is None:
    print("CFG.winning_backbone is unset — fill it in from Stage 2's "
          "results/backbone_sweep_summary.csv before running Section 3.")
print(f"{len(CFG.videos)} videos, {N_CLASSES} phase classes, "
      f"winning_run_name={CFG.winning_run_name!r}, "
      f"raw duration budget for clip selection: {RAW_DURATION_RANGE[0]:.0f}-{RAW_DURATION_RANGE[1]:.0f}s")


## 2. Preprocessing — strip audio

Every raw video is copied once, with `ffmpeg -an` dropping the audio
stream entirely (`-c:v copy` re-muxes the video stream without
re-encoding, so this is fast and lossless). All later sections — clip
extraction, reel assembly — read exclusively from these audio-free copies
in `DIRS['no_audio_videos']`, never from `DIRS['videos']` directly. This is
belt-and-braces: no audio ever reaches any intermediate file, let alone a
final reel.


In [ ]:
def strip_audio(video_id: str, overwrite: bool = False) -> Path:
    src = DIRS["videos"] / f"{video_id}.mp4"
    dst = DIRS["no_audio_videos"] / f"{video_id}.mp4"
    if not src.exists():
        raise FileNotFoundError(f"Raw video not found: {src}")
    if dst.exists() and not overwrite:
        return dst
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src), "-an", "-c:v", "copy", str(dst)],
        check=True, capture_output=True,
    )
    return dst


def strip_audio_all_videos(overwrite: bool = False):
    for video_id in tqdm(CFG.videos, desc="stripping audio"):
        strip_audio(video_id, overwrite=overwrite)
    print(f"Audio-free copies in {DIRS['no_audio_videos']}")


def no_audio_path(video_id: str) -> Path:
    p = DIRS["no_audio_videos"] / f"{video_id}.mp4"
    if not p.exists():
        raise FileNotFoundError(f"{p} missing — run strip_audio_all_videos() first.")
    return p


# strip_audio_all_videos()


## 3. Load Stage 2/Stage 1 artefacts

Nothing in this section computes anything new — it only reads what Stage 2
(phase manifest, per-frame visual embeddings, per-fold GRU checkpoints) and
Stage 1 (per-frame instrument detections) already saved to disk.


In [ ]:
def load_manifest() -> pd.DataFrame:
    path = DIRS["manifest"] / "frames_manifest.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} missing — run Stage 2's build_full_manifest() first."
        )
    return pd.read_csv(path, dtype={"video_id": str})


def load_cached_features(backbone_name: str, video_id: str) -> np.ndarray:
    path = DIRS["features"] / backbone_name / f"{video_id}.npy"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} missing — run Stage 2's extract_all_backbones() first."
        )
    return np.load(path)


DETECTOR_FEATURE_COLS = [
    f"{c}_{f}" for c in CFG.detector_classes for f in ("confidence", "present", "count")
]


def load_detector_features(video_id: str) -> pd.DataFrame:
    '''Expects DIRS['detector']/{video_id}.csv, written by Stage 2's
    run_avos_detector_on_videos. Only used here for captioning ("instruments
    detected") — never as a saliency or selection input for Reel 2, which
    must stay saliency-blind and, per its own spec, could in principle stay
    detector-blind too; it isn't, because captions describing what's on
    screen are a legitimate substitute for a human glancing at the frame.'''
    path = DIRS["detector"] / f"{video_id}.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} missing — run Stage 2's detector pass first.")
    df = pd.read_csv(path)
    return df.reindex(columns=["frame_idx"] + DETECTOR_FEATURE_COLS, fill_value=0.0)


def instruments_present_at(detector_row: pd.Series, min_confidence: float = 0.25) -> list:
    return [
        cls for cls in CFG.detector_classes
        if detector_row.get(f"{cls}_present", 0) and detector_row.get(f"{cls}_confidence", 0) >= min_confidence
    ]


### 3.1 Reload the winning phase-recognition model

`PhaseGRU`, `make_windows`, and `load_fold_checkpoint` are copied verbatim
from Stage 2 (this notebook is standalone, per this repo's convention of
one self-contained notebook per stage) — the class definition has to match
exactly for `load_state_dict` to succeed. `build_fold_scalers` reproduces
Stage 2 Section 7.1's per-fold standardisation: **the scaler was fit at
train time on that fold's six training videos and never saved**, so Stage 4
refits it the same way, from the same six videos, which is deterministic
and gives back the identical transform.


In [ ]:
from sklearn.preprocessing import StandardScaler


def loocv_folds():
    for fold_idx, held_out in enumerate(CFG.videos):
        train_videos = [v for v in CFG.videos if v != held_out]
        yield fold_idx, held_out, train_videos


class PhaseGRU(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, n_classes: int = N_CLASSES):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=1, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden_dim * 2, n_classes)

    def forward(self, x):
        x = self.proj(x)
        x, _ = self.gru(x)
        return self.head(x)


def make_windows(seq_len: int, window_size: int, stride: int):
    if seq_len <= window_size:
        return [0]
    starts = list(range(0, seq_len - window_size + 1, stride))
    if starts[-1] + window_size < seq_len:
        starts.append(seq_len - window_size)
    return starts


def load_fold_checkpoint(run_name: str, video_id: str) -> "tuple[PhaseGRU, dict]":
    '''The one model legitimately usable for `video_id`: the LOOCV fold of
    `run_name` that held it out. Refuses to load if the checkpoint's own
    metadata shows `video_id` was in that fold's training set.'''
    ckpt_stem = f"{run_name}__{video_id}"
    meta_path = FOLD_CKPT_DIR / f"{ckpt_stem}.json"
    weights_path = FOLD_CKPT_DIR / f"{ckpt_stem}.pt"
    if not meta_path.exists() or not weights_path.exists():
        raise FileNotFoundError(
            f"No checkpoint for run={run_name!r}, held-out video={video_id!r} at {weights_path}. "
            "Did Stage 2's run_gru_loocv finish for this run?"
        )
    with open(meta_path) as f:
        meta = json.load(f)
    if meta["held_out_video"] != video_id or video_id in meta["train_videos"]:
        raise AssertionError(
            f"Checkpoint metadata mismatch for {weights_path}: expected held-out video "
            f"{video_id!r}, but metadata says held_out_video={meta['held_out_video']!r}. "
            f"Refusing to load — using this checkpoint would let {video_id!r} leak into its own reel."
        )
    model = PhaseGRU(input_dim=meta["input_dim"], hidden_dim=meta["hidden_dim"], n_classes=meta["n_classes"])
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.to(DEVICE).eval()
    return model, meta


def build_fold_input(video_id: str, train_videos: list, backbone_name: str, use_detector: bool) -> np.ndarray:
    '''Standardised model input for `video_id`, using scalers fit on
    `train_videos` only — reproduces Stage 2 Section 7.1 exactly. Returns
    (seq_len, input_dim); the first `backbone_dim` columns are always the
    (standardised) visual embedding, which is what compute_frame_saliency
    differentiates with respect to.'''
    backbone_cache = {vid: load_cached_features(backbone_name, vid) for vid in CFG.videos}
    backbone_scaler = StandardScaler().fit(np.concatenate([backbone_cache[v] for v in train_videos]))
    feats = backbone_scaler.transform(backbone_cache[video_id])
    backbone_dim = feats.shape[1]

    if use_detector:
        detector_cache = {
            vid: load_detector_features(vid)[DETECTOR_FEATURE_COLS].values.astype(np.float32)
            for vid in CFG.videos
        }
        detector_scaler = StandardScaler().fit(np.concatenate([detector_cache[v] for v in train_videos]))
        det = detector_scaler.transform(detector_cache[video_id][: len(feats)])
        feats = np.concatenate([feats, det], axis=1)

    return feats.astype(np.float32), backbone_dim


### 3.2 Gradient-based saliency

For each frame, saliency is the gradient of the model's own predicted-phase
logit with respect to that frame's (standardised) visual embedding —
vanilla gradient saliency (Simonyan et al.), applied per frame rather than
per whole-image, and always evaluated through the fold checkpoint that
held the video out so the saliency reflects genuine held-out behaviour.

**Design decision — one backward pass per window, not one per frame.** An
exact per-frame Jacobian (∂logit_t/∂x_t for every t) would need one
backward pass per timestep because of the GRU's recurrence, which is
O(window_size) backward passes per window — for the sequence lengths here
that's tractable but very slow for no real benefit: this notebook instead
backpropagates the **sum** of every timestep's own predicted-class logit in
one pass, `loss = sum_t logits[t, argmax_t]`, and reads off `d(loss)/dx_j`
at position j. Cross-terms (`d(logit_t)/dx_j` for `t != j`) are included in
that sum, but a GRU's hidden state at time t is dominated by nearby inputs,
so this is a close, much cheaper proxy for the diagonal term, applied
uniformly across all frames and all videos. If a future user needs the
exact per-frame Jacobian, `exact=True` below does the O(window_size) loop
instead.


In [ ]:
def compute_frame_saliency(video_id: str, exact: bool = False) -> pd.DataFrame:
    manifest = load_manifest()
    vm = manifest[manifest.video_id == video_id].sort_values("frame_idx").reset_index(drop=True)

    train_videos = [v for v in CFG.videos if v != video_id]
    use_detector = CFG.winning_run_name == "model_D"
    feats, backbone_dim = build_fold_input(video_id, train_videos, CFG.winning_backbone, use_detector)
    feats = feats[: len(vm)]
    seq_len = len(feats)

    model, meta = load_fold_checkpoint(CFG.winning_run_name, video_id)

    saliency_sum = np.zeros(seq_len, dtype=np.float64)
    logit_sum = np.zeros((seq_len, N_CLASSES), dtype=np.float64)
    count = np.zeros(seq_len, dtype=np.int32)

    for start in make_windows(seq_len, CFG.window_size, CFG.window_stride):
        end = min(start + CFG.window_size, seq_len)
        window = feats[start:end]
        pad = CFG.window_size - len(window)
        if pad > 0:
            window = np.pad(window, ((0, pad), (0, 0)))

        x = torch.from_numpy(window).float().unsqueeze(0).to(DEVICE)
        x.requires_grad_(True)
        logits = model(x)[0]  # (window_size, n_classes)

        valid = end - start
        target_idx = logits[:valid].argmax(dim=-1).detach()
        logit_sum[start:end] += logits[:valid].detach().cpu().numpy()

        # Restrict every gradient read below to the visual-embedding
        # sub-block of the input — columns [0, backbone_dim). For
        # winning_run_name == "model_D" this excludes the concatenated
        # detector-feature columns, so saliency stays "gradient w.r.t.
        # visual embeddings" even when the winning model also consumes
        # detector features.
        if exact:
            grad_norms = np.zeros(valid, dtype=np.float64)
            for t in range(valid):
                model.zero_grad(set_to_none=True)
                if x.grad is not None:
                    x.grad = None
                logits[t, target_idx[t]].backward(retain_graph=True)
                grad_norms[t] = x.grad[0, t, :backbone_dim].detach().cpu().norm().item()
            saliency_sum[start:end] += grad_norms
        else:
            target_logits = logits[:valid].gather(-1, target_idx.unsqueeze(-1)).squeeze(-1)
            model.zero_grad(set_to_none=True)
            target_logits.sum().backward()
            grad_norms = x.grad[0, :valid, :backbone_dim].detach().cpu().norm(dim=-1).numpy()
            saliency_sum[start:end] += grad_norms

        count[start:end] += 1

    avg_logits = logit_sum / count[:, None]
    avg_saliency = saliency_sum / count

    out = pd.DataFrame({
        "video_id": video_id,
        "frame_idx": vm.frame_idx.values,
        "timestamp": vm.timestamp.values,
        "predicted_phase_idx": avg_logits.argmax(axis=1),
        "saliency_raw": avg_saliency,
    })
    out["predicted_phase"] = out.predicted_phase_idx.map(IDX_TO_PHASE)
    return out


def compute_all_frame_saliency(exact: bool = False) -> pd.DataFrame:
    parts = []
    for video_id in tqdm(CFG.videos, desc="saliency"):
        df = compute_frame_saliency(video_id, exact=exact)
        df.to_csv(DIRS["saliency"] / f"{video_id}_raw.csv", index=False)
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


# saliency_raw = compute_all_frame_saliency()


### 3.3 Min-max normalisation and smoothing (per video)

Saliency scale is not comparable across videos (different footage, different
gradient magnitudes), so normalisation is always **within** a single video.
Smoothing uses a centred rolling mean over `saliency_smooth_window_s`
seconds (converted to frames via `base_fps`) purely to reduce frame-level
noise before thresholding — thresholding directly on the raw signal would
fragment single surgical moments into many tiny segments.


In [ ]:
def normalize_and_smooth(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    lo, hi = df.saliency_raw.min(), df.saliency_raw.max()
    df["saliency_norm"] = (df.saliency_raw - lo) / (hi - lo) if hi > lo else 0.0

    smooth_window_frames = max(1, round(CFG.saliency_smooth_window_s * CFG.base_fps))
    df["saliency_smooth"] = (
        df.saliency_norm.rolling(smooth_window_frames, center=True, min_periods=1).mean()
    )
    return df


def normalize_and_smooth_all(saliency_raw: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for video_id, group in saliency_raw.groupby("video_id"):
        df = normalize_and_smooth(group.sort_values("frame_idx"))
        df.to_csv(DIRS["saliency"] / f"{video_id}_smoothed.csv", index=False)
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


# saliency = normalize_and_smooth_all(saliency_raw)


### 3.4 Threshold sweep and contiguous segment extraction

`saliency_thresholds` is swept purely to characterise the trade-off (how
many segments, how much total duration, at each cut) before committing to
`saliency_operating_threshold` for the candidate pools built in Sections 4
and 7. Segments shorter than `min_segment_s` after thresholding are dropped
as noise rather than kept as real surgical moments.


In [ ]:
def frame_diffs_to_segments(mask: np.ndarray):
    '''Start/end indices (end exclusive) of contiguous True runs in mask.'''
    segments = []
    start = None
    for i, v in enumerate(mask):
        if v and start is None:
            start = i
        elif not v and start is not None:
            segments.append((start, i))
            start = None
    if start is not None:
        segments.append((start, len(mask)))
    return segments


def extract_segments(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    df = df.sort_values("frame_idx").reset_index(drop=True)
    mask = (df.saliency_smooth >= threshold).values
    rows = []
    for start, end in frame_diffs_to_segments(mask):
        seg = df.iloc[start:end]
        duration = seg.timestamp.iloc[-1] - seg.timestamp.iloc[0]
        if duration < CFG.min_segment_s:
            continue
        phase_mode = seg.predicted_phase.mode()
        rows.append({
            "video_id": seg.video_id.iloc[0],
            "start_frame_idx": seg.frame_idx.iloc[0],
            "end_frame_idx": seg.frame_idx.iloc[-1],
            "start_ts": seg.timestamp.iloc[0],
            "end_ts": seg.timestamp.iloc[-1],
            "duration_s": duration,
            "phase": phase_mode.iloc[0] if len(phase_mode) else seg.predicted_phase.iloc[0],
            "mean_saliency": seg.saliency_smooth.mean(),
            "peak_saliency": seg.saliency_smooth.max(),
            "threshold": threshold,
        })
    return pd.DataFrame(rows)


def sweep_thresholds(saliency: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for threshold in CFG.saliency_thresholds:
        for video_id, group in saliency.groupby("video_id"):
            segs = extract_segments(group, threshold)
            rows.append({
                "threshold": threshold,
                "video_id": video_id,
                "n_segments": len(segs),
                "total_duration_s": segs.duration_s.sum() if len(segs) else 0.0,
                "mean_segment_s": segs.duration_s.mean() if len(segs) else 0.0,
            })
    summary = pd.DataFrame(rows)
    summary.to_csv(DIRS["saliency"] / "threshold_sweep_summary.csv", index=False)
    display(summary.pivot(index="video_id", columns="threshold", values="n_segments"))
    return summary


def build_candidate_pool(saliency: pd.DataFrame, threshold: float = None) -> pd.DataFrame:
    threshold = CFG.saliency_operating_threshold if threshold is None else threshold
    parts = [extract_segments(group, threshold) for _, group in saliency.groupby("video_id")]
    pool = pd.concat(parts, ignore_index=True)
    pool.to_csv(DIRS["candidates"] / f"saliency_pool_tau{threshold}.csv", index=False)
    print(f"{len(pool)} candidate segments across {pool.video_id.nunique()} videos at tau={threshold}")
    return pool


# threshold_sweep = sweep_thresholds(saliency)
# saliency_candidates = build_candidate_pool(saliency)


## 4. Reel 1 — saliency only

Clips come directly from the saliency-thresholded candidate pool (Section
3.4), ranked within each phase by saliency and taken top-N, with no LLM
involvement and no captioning. **Reel 1 runs first because it sets the
clips-per-phase quota Reels 2 and 3 are asked to match** — N is grown
uniformly across phases until adding another round would push the raw
footage budget (`RAW_DURATION_RANGE`) over its ceiling.


In [ ]:
def trim_to_max_duration(df: pd.DataFrame, max_s: float, sort_col: str = "mean_saliency", ascending: bool = True) -> pd.DataFrame:
    '''Drop the lowest-priority clips first (lowest `sort_col` when
    ascending=True — e.g. lowest saliency, or highest LLM rank number
    when called with sort_col="llm_rank", ascending=False), keeping at
    least one clip per phase where possible, until total duration fits
    under max_s.'''
    keep = df.copy()
    for idx, row in df.sort_values(sort_col, ascending=ascending).iterrows():
        if keep.duration_s.sum() <= max_s:
            break
        if (keep.phase == row.phase).sum() <= 1:
            continue
        keep = keep.drop(idx)
    return keep


def select_top_n_per_phase(pool: pd.DataFrame, target_range: tuple):
    '''Grows N (segments kept per phase, ranked by mean_saliency
    descending) until including one more round from every phase would
    exceed target_range[1]. Returns (selected_df, n_per_phase, achieved_n).
    '''
    ranked = {
        phase: group.sort_values("mean_saliency", ascending=False).reset_index(drop=True)
        for phase, group in pool.groupby("phase")
    }
    if not ranked:
        return pool.iloc[0:0], {}, 0

    n = 1
    best = None
    while True:
        rounds = [g.iloc[:n] for g in ranked.values()]
        candidate = pd.concat(rounds, ignore_index=True)
        duration = candidate.duration_s.sum()
        if duration > target_range[1]:
            if best is None:
                best = trim_to_max_duration(candidate, target_range[1])
                n_per_phase = {phase: min(n, len(g)) for phase, g in ranked.items()}
            break
        best = candidate
        n_per_phase = {phase: min(n, len(g)) for phase, g in ranked.items()}
        if all(n >= len(g) for g in ranked.values()):
            break
        n += 1

    if best.duration_s.sum() < target_range[0]:
        print(f"  Warning: even using every available candidate segment, total duration "
              f"{best.duration_s.sum():.0f}s is below the {target_range[0]:.0f}s floor — "
              "this video's saliency pool is too sparse to hit the target unassisted.")
    return best, n_per_phase, n


def build_reel1(video_id: str, saliency_candidates: pd.DataFrame):
    pool = saliency_candidates[saliency_candidates.video_id == video_id]
    clips, n_per_phase, n = select_top_n_per_phase(pool, RAW_DURATION_RANGE)
    clips = clips.sort_values("start_ts").reset_index(drop=True)
    clips.to_csv(DIRS["selections"] / f"reel1_{video_id}.csv", index=False)
    with open(DIRS["selections"] / f"clips_per_phase_{video_id}.json", "w") as f:
        json.dump(n_per_phase, f, indent=2)
    print(f"{video_id}: Reel 1 — {len(clips)} clips, {clips.duration_s.sum():.0f}s raw "
          f"({clips.duration_s.sum() / CFG.playback_speed:.0f}s played back), n_per_phase={n_per_phase}")
    return clips, n_per_phase


def build_reel1_all(saliency_candidates: pd.DataFrame):
    reels, quotas = {}, {}
    for video_id in CFG.videos:
        clips, n_per_phase = build_reel1(video_id, saliency_candidates)
        reels[video_id] = clips
        quotas[video_id] = n_per_phase
    return reels, quotas


# reel1_clips, clips_per_phase_quota = build_reel1_all(saliency_candidates)


## 5. Templated captioning

Frame-level captions are generated from visual content only — the model's
own predicted phase (Section 3.2's `predicted_phase`, so this stays
consistent with whatever the saliency computation already used) plus
Stage 1's detected instruments — using a fixed string template, never an
LLM. Frame captions are then aggregated into clip-level captions two ways:
a **sliding window** over the whole video, at fixed intervals independent
of saliency (this *is* Reel 2's candidate pool — Section 6), and an
aggregation **over an arbitrary span**, used for Reel 3's saliency segments,
which have irregular lengths.


In [ ]:
def format_timestamp(seconds: float) -> str:
    seconds = max(0, int(round(seconds)))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def build_frame_captions(video_id: str, saliency: pd.DataFrame) -> pd.DataFrame:
    vm = saliency[saliency.video_id == video_id].sort_values("frame_idx").reset_index(drop=True)
    detector = load_detector_features(video_id)
    merged = vm.merge(detector, on="frame_idx", how="left")

    instruments_col, captions = [], []
    for _, row in merged.iterrows():
        instruments = instruments_present_at(row)
        instr_text = ", ".join(instruments) if instruments else "none detected"
        instruments_col.append(instruments)
        captions.append(
            f"[{format_timestamp(row.timestamp)}] Phase: {row.predicted_phase}. "
            f"Instruments in view: {instr_text}."
        )
    merged["instruments"] = instruments_col
    merged["frame_caption"] = captions
    return merged[["video_id", "frame_idx", "timestamp", "predicted_phase", "instruments", "frame_caption"]]


def build_frame_captions_all(saliency: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for video_id in CFG.videos:
        df = build_frame_captions(video_id, saliency)
        df.to_csv(DIRS["captions"] / f"frame_captions_{video_id}.csv", index=False)
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


# frame_captions = build_frame_captions_all(saliency)


### 5.1 Aggregating frame captions into a clip caption

Shared by both the sliding-window (Reel 2) and arbitrary-span (Reel 3)
routes below — a clip caption reports the clip's time range, its dominant
predicted phase (and whether it actually spans a phase transition, since
that matters for "no redundancy" judgements), and which instruments
appeared and in what fraction of frames.


In [ ]:
def aggregate_frame_captions(frames: pd.DataFrame, start_ts: float, end_ts: float) -> str:
    if len(frames) == 0:
        return f"[{format_timestamp(start_ts)}-{format_timestamp(end_ts)}] No frames captioned in this span."

    phase_counts = frames.predicted_phase.value_counts()
    dominant_phase = phase_counts.idxmax()
    transition_note = (
        "" if len(phase_counts) == 1
        else f" Also touches: {', '.join(p for p in phase_counts.index if p != dominant_phase)} (phase transition)."
    )

    instrument_fracs = {}
    for cls in CFG.detector_classes:
        frac = frames.instruments.apply(lambda lst: cls in lst).mean()
        if frac > 0:
            instrument_fracs[cls] = frac
    if instrument_fracs:
        instr_text = ", ".join(f"{cls} ({frac:.0%} of frames)" for cls, frac in instrument_fracs.items())
    else:
        instr_text = "none detected"

    duration = end_ts - start_ts
    return (
        f"[{format_timestamp(start_ts)}-{format_timestamp(end_ts)}] ({duration:.0f}s) "
        f"Primarily phase '{dominant_phase}' ({phase_counts.max() / len(frames):.0%} of frames)."
        f"{transition_note} Instruments detected: {instr_text}."
    )


def build_caption_windows(frame_captions: pd.DataFrame, video_id: str, window_s: float, overlap: float) -> pd.DataFrame:
    '''Reel 2's candidate pool: fixed-size, fixed-stride windows over the
    whole video, independent of saliency. stride = window_s * (1 - overlap).
    '''
    vm = frame_captions[frame_captions.video_id == video_id].sort_values("timestamp")
    if len(vm) == 0:
        return pd.DataFrame()
    total_duration = vm.timestamp.iloc[-1]
    stride = window_s * (1 - overlap)

    rows = []
    start = 0.0
    clip_id = 0
    while start < total_duration:
        end = min(start + window_s, total_duration)
        frames = vm[(vm.timestamp >= start) & (vm.timestamp < end)]
        dominant_phase = frames.predicted_phase.mode()
        rows.append({
            "video_id": video_id,
            "clip_id": f"{video_id}_w{window_s:.0f}_{clip_id}",
            "start_ts": start,
            "end_ts": end,
            "duration_s": end - start,
            "phase": dominant_phase.iloc[0] if len(dominant_phase) else "background",
            "caption": aggregate_frame_captions(frames, start, end),
        })
        clip_id += 1
        start += stride
    return pd.DataFrame(rows)


def build_caption_for_segment(frame_captions: pd.DataFrame, video_id: str, start_ts: float, end_ts: float) -> str:
    vm = frame_captions[frame_captions.video_id == video_id]
    frames = vm[(vm.timestamp >= start_ts) & (vm.timestamp <= end_ts)]
    return aggregate_frame_captions(frames, start_ts, end_ts)


### 5.2 Comparing captioning window sizes (512s vs 32s, 50% overlap)

Both window sizes in `caption_window_s_list` are built and compared here —
number of candidate clips and their duration — purely as an ablation.
`caption_window_primary_s` is the one that actually feeds Reel 2 and gets
assembled and evaluated end-to-end below.


In [ ]:
def compare_caption_window_sizes(frame_captions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for window_s in CFG.caption_window_s_list:
        for video_id in CFG.videos:
            windows = build_caption_windows(frame_captions, video_id, window_s, CFG.caption_window_overlap)
            rows.append({
                "window_s": window_s,
                "video_id": video_id,
                "n_candidate_clips": len(windows),
                "mean_duration_s": windows.duration_s.mean() if len(windows) else 0.0,
            })
    summary = pd.DataFrame(rows)
    summary.to_csv(DIRS["captions"] / "caption_window_size_comparison.csv", index=False)
    display(summary.pivot(index="video_id", columns="window_s", values="n_candidate_clips"))
    return summary


# caption_window_comparison = compare_caption_window_sizes(frame_captions)


## 6. LLM backends and the shared selection prompt

Reel 2 and Reel 3 share everything downstream of "here is a list of
captioned candidate clips" — the instructions, the JSON response format,
the three backend wrappers, and the parsing/validation/duration-fitting
logic. What differs between them is only *which candidate pool* gets
handed in (Section 6.1 vs Section 7).

The selection instructions given to the LLM cover exactly the four things
specified: **phase coverage** (every phase actually present in the video
should be represented, aiming for `phase_quota` clips per phase — the same
quota Reel 1 needed to hit its own duration target, so all three reels are
matched on clips per phase), **no redundancy** (don't pick two clips that
say the same thing about the same phase), **clinical priority** (prioritise
clinically pivotal steps — incisions, tourniquet application/release,
urethroplasty, glansplasty — over administrative or setup steps such as
dressing or preoperative anatomy, when a choice must be made), and the
**duration constraint**.


In [ ]:
def build_selection_prompt(candidates: pd.DataFrame, phase_quota: dict, raw_duration_range: tuple) -> str:
    final_range = (raw_duration_range[0] / CFG.playback_speed, raw_duration_range[1] / CFG.playback_speed)
    quota_text = ", ".join(f"{phase}: {n}" for phase, n in phase_quota.items()) or "no quota available"

    lines = [
        "You are assisting with building a surgical highlight reel for teaching purposes.",
        "Below is a list of candidate video clips from one surgery, each with a clip_id and a caption",
        "describing what happens in it (phase and instruments in view). No video or images are given —",
        "base your selection solely on the caption text.",
        "",
        "Select a subset of these clips for the highlight reel. Requirements:",
        f"1. Phase coverage: represent every phase you see mentioned, aiming for this many clips per",
        f"   phase where available: {quota_text}.",
        "2. No redundancy: do not select two clips that describe essentially the same moment of the",
        "   same phase with no new information.",
        "3. Clinical priority: when you must choose between clips, prefer clinically pivotal steps",
        "   (incisions, tourniquet application/release, urethroplasty, glansplasty) over administrative",
        "   or setup steps (dressing, preoperative anatomy) if it means staying within the duration budget.",
        f"4. Duration: the reel is played back at {CFG.playback_speed:.0f}x speed, so it should end up",
        f"   {final_range[0]:.0f}-{final_range[1]:.0f} seconds long once sped up — meaning your selected",
        f"   clips' *original* durations should sum to roughly {raw_duration_range[0]:.0f}-{raw_duration_range[1]:.0f} seconds.",
        "",
        "Candidate clips:",
    ]
    for _, row in candidates.iterrows():
        lines.append(f"- clip_id={row.clip_id} | duration={row.duration_s:.0f}s | {row.caption}")
    lines += [
        "",
        "Respond with ONLY a JSON array, no other text, where each element is",
        '{"clip_id": "...", "rationale": "one sentence on why this clip was chosen"}.',
    ]
    return "\n".join(lines)


def call_chatgpt(prompt: str) -> str:
    if openai is None:
        raise ImportError("pip install openai")
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.chat.completions.create(
        model=CFG.openai_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content


def call_claude(prompt: str) -> str:
    if anthropic is None:
        raise ImportError("pip install anthropic")
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    resp = client.messages.create(
        model=CFG.anthropic_model,
        max_tokens=4096,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(block.text for block in resp.content if block.type == "text")


def call_ollama_backend(prompt: str) -> str:
    if ollama is None:
        raise ImportError("pip install ollama")
    client = ollama.Client(host=CFG.ollama_host)
    resp = client.chat(model=CFG.ollama_model, messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"]


LLM_CALLERS = {"chatgpt": call_chatgpt, "claude": call_claude, "ollama": call_ollama_backend}


def call_llm(backend: str, prompt: str) -> str:
    caller = LLM_CALLERS[backend]
    last_error = None
    for attempt in range(CFG.llm_max_retries):
        try:
            return caller(prompt)
        except Exception as e:
            last_error = e
            print(f"  {backend} attempt {attempt + 1}/{CFG.llm_max_retries} failed: {e}")
            time.sleep(2 ** attempt)
    raise RuntimeError(f"{backend} failed after {CFG.llm_max_retries} attempts") from last_error


def parse_llm_selection(response_text: str, candidates: pd.DataFrame) -> "tuple[pd.DataFrame, dict]":
    match = re.search(r"\[.*\]", response_text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON array found in LLM response: {response_text[:300]!r}")
    items = json.loads(match.group(0))

    known_ids = set(candidates.clip_id)
    rationale = {}
    ordered_ids = []
    for rank, item in enumerate(items):
        clip_id = item.get("clip_id")
        if clip_id not in known_ids:
            print(f"  Dropping hallucinated clip_id from LLM response: {clip_id!r}")
            continue
        if clip_id in rationale:
            continue  # duplicate selection, keep first (highest-ranked) occurrence
        rationale[clip_id] = item.get("rationale", "")
        ordered_ids.append(clip_id)

    selected = candidates[candidates.clip_id.isin(ordered_ids)].copy()
    selected["llm_rank"] = selected.clip_id.map({cid: i for i, cid in enumerate(ordered_ids)})
    selected = selected.sort_values("llm_rank").reset_index(drop=True)
    return selected, rationale


def fit_llm_selection_to_budget(selected: pd.DataFrame, raw_duration_range: tuple) -> pd.DataFrame:
    if selected.duration_s.sum() > raw_duration_range[1]:
        selected = trim_to_max_duration(selected, raw_duration_range[1], sort_col="llm_rank", ascending=False)
    if selected.duration_s.sum() < raw_duration_range[0]:
        print(f"  Warning: LLM selection totals {selected.duration_s.sum():.0f}s raw, "
              f"below the {raw_duration_range[0]:.0f}s floor — accepting as-is rather than padding "
              "with clips the LLM did not choose.")
    return selected


### 6.1 Reel 2 — LLM only

Candidate clips are the sliding-window captions from Section 5
(`caption_window_primary_s`, evenly spaced, independent of saliency — no
saliency score is computed, used, or shown to the LLM anywhere in this
path). The LLM sees only captions and picks a subset.


In [ ]:
def load_phase_quota(video_id: str) -> dict:
    path = DIRS["selections"] / f"clips_per_phase_{video_id}.json"
    if not path.exists():
        raise FileNotFoundError(f"{path} missing — run build_reel1_all() first (Reel 2/3 match its quota).")
    with open(path) as f:
        return json.load(f)


def build_reel2(video_id: str, backend: str, frame_captions: pd.DataFrame) -> "tuple[pd.DataFrame, dict]":
    candidates = build_caption_windows(frame_captions, video_id, CFG.caption_window_primary_s, CFG.caption_window_overlap)
    candidates.to_csv(DIRS["candidates"] / f"reel2_candidates_{video_id}.csv", index=False)

    phase_quota = load_phase_quota(video_id)
    prompt = build_selection_prompt(candidates, phase_quota, RAW_DURATION_RANGE)
    response = call_llm(backend, prompt)
    selected, rationale = parse_llm_selection(response, candidates)
    selected = fit_llm_selection_to_budget(selected, RAW_DURATION_RANGE)
    selected = selected.sort_values("start_ts").reset_index(drop=True)

    selected.to_csv(DIRS["selections"] / f"reel2_{backend}_{video_id}.csv", index=False)
    with open(DIRS["selections"] / f"reel2_{backend}_{video_id}_rationale.json", "w") as f:
        json.dump(rationale, f, indent=2)
    print(f"{video_id}/{backend}: Reel 2 — {len(selected)} clips, {selected.duration_s.sum():.0f}s raw")
    return selected, rationale


def build_reel2_all(frame_captions: pd.DataFrame):
    reels = {}
    for backend in CFG.llm_backends:
        for video_id in CFG.videos:
            try:
                clips, rationale = build_reel2(video_id, backend, frame_captions)
                reels[(backend, video_id)] = clips
            except Exception as e:
                print(f"Skipping Reel 2 for {backend}/{video_id}: {e}")
    return reels


# reel2_clips = build_reel2_all(frame_captions)


## 7. Reel 3 — combined

Candidate pool is the *same* saliency-thresholded segments used for Reel 1
(Section 3.4's `saliency_candidates`), captioned with the same templated
approach as Reel 2 — but aggregated over each segment's own (irregular)
span rather than a fixed sliding window, since these candidates already
have data-driven boundaries. The LLM then re-ranks and selects from this
saliency-shortlisted pool under the identical instructions used for Reel 2,
so any difference between Reel 2 and Reel 3's LLM picks reflects the
saliency pre-filter, not a change in what the LLM was asked to do.


In [ ]:
def build_reel3(video_id: str, backend: str, saliency_candidates: pd.DataFrame,
                 frame_captions: pd.DataFrame) -> "tuple[pd.DataFrame, dict]":
    pool = saliency_candidates[saliency_candidates.video_id == video_id].copy()
    pool["clip_id"] = [f"{video_id}_sal_{i}" for i in range(len(pool))]
    pool["caption"] = [
        build_caption_for_segment(frame_captions, video_id, row.start_ts, row.end_ts)
        for row in pool.itertuples()
    ]
    pool.to_csv(DIRS["candidates"] / f"reel3_candidates_{video_id}.csv", index=False)

    phase_quota = load_phase_quota(video_id)
    prompt = build_selection_prompt(pool, phase_quota, RAW_DURATION_RANGE)
    response = call_llm(backend, prompt)
    selected, rationale = parse_llm_selection(response, pool)
    selected = fit_llm_selection_to_budget(selected, RAW_DURATION_RANGE)
    selected = selected.sort_values("start_ts").reset_index(drop=True)

    selected.to_csv(DIRS["selections"] / f"reel3_{backend}_{video_id}.csv", index=False)
    with open(DIRS["selections"] / f"reel3_{backend}_{video_id}_rationale.json", "w") as f:
        json.dump(rationale, f, indent=2)
    print(f"{video_id}/{backend}: Reel 3 — {len(selected)} clips, {selected.duration_s.sum():.0f}s raw")
    return selected, rationale


def build_reel3_all(saliency_candidates: pd.DataFrame, frame_captions: pd.DataFrame):
    reels = {}
    for backend in CFG.llm_backends:
        for video_id in CFG.videos:
            try:
                clips, rationale = build_reel3(video_id, backend, saliency_candidates, frame_captions)
                reels[(backend, video_id)] = clips
            except Exception as e:
                print(f"Skipping Reel 3 for {backend}/{video_id}: {e}")
    return reels


# reel3_clips = build_reel3_all(saliency_candidates, frame_captions)


## 8. Assembly

Shared by all three reels. Each selected clip is cut from the audio-free
source (Section 2) with a burned-in phase-label overlay and 2x playback
speed applied in the same `ffmpeg` pass, then the clips are concatenated
**in chronological order by original timestamp** — selection strategy
decides which moments make the reel, but the reel always replays them in
the order they actually happened. `-an` is passed explicitly at every
step even though the source already carries no audio, and encoding uses
`video_crf` (0 = lossless by default) so quality is never traded away for
file size without an explicit config change.


In [ ]:
def _ffmpeg_escape_text(text: str) -> str:
    return text.replace("\\", "\\\\").replace(":", "\\:").replace("'", "\\'")


def _drawtext_font() -> str:
    return fm.findfont(fm.FontProperties(family="DejaVu Sans"))


def extract_and_label_clip(video_id: str, start_ts: float, end_ts: float, label: str, out_path: Path):
    src = no_audio_path(video_id)
    label_text = _ffmpeg_escape_text(label)
    vf = (
        f"drawtext=fontfile='{_drawtext_font()}':text='{label_text}':"
        f"x=20:y=h-th-20:fontsize={CFG.overlay_font_size}:fontcolor=white:"
        f"box=1:boxcolor=black@0.5:boxborderw=8,"
        f"setpts={1 / CFG.playback_speed}*PTS"
    )
    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start_ts), "-to", str(end_ts), "-i", str(src),
        "-an", "-vf", vf,
        "-c:v", "libx264", "-crf", str(CFG.video_crf), "-preset", CFG.video_preset,
        str(out_path),
    ], check=True, capture_output=True)


def assemble_reel(video_id: str, clips: pd.DataFrame, reel_name: str) -> Path:
    if len(clips) == 0:
        raise ValueError(f"No clips to assemble for {reel_name}/{video_id}")
    clips = clips.sort_values("start_ts").reset_index(drop=True)

    tmp_dir = DIRS["reels"] / f"_tmp_{reel_name}_{video_id}"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    clip_paths = []
    for i, row in clips.iterrows():
        clip_path = tmp_dir / f"clip_{i:03d}.mp4"
        extract_and_label_clip(video_id, row.start_ts, row.end_ts, row.phase, clip_path)
        clip_paths.append(clip_path)

    concat_list = tmp_dir / "concat_list.txt"
    with open(concat_list, "w") as f:
        for p in clip_paths:
            f.write(f"file '{p.resolve()}'\n")

    out_path = DIRS["reels"] / f"{reel_name}_{video_id}.mp4"
    subprocess.run([
        "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(concat_list),
        "-an", "-c", "copy", str(out_path),
    ], check=True, capture_output=True)

    shutil.rmtree(tmp_dir)
    print(f"{video_id}: assembled {reel_name} -> {out_path} "
          f"({clips.duration_s.sum() / CFG.playback_speed:.0f}s played back)")
    return out_path


def assemble_all_reels(reel1_clips: dict, reel2_clips: dict, reel3_clips: dict):
    paths = {}
    for video_id, clips in reel1_clips.items():
        paths[("reel1", None, video_id)] = assemble_reel(video_id, clips, "reel1")
    for (backend, video_id), clips in reel2_clips.items():
        paths[("reel2", backend, video_id)] = assemble_reel(video_id, clips, f"reel2_{backend}")
    for (backend, video_id), clips in reel3_clips.items():
        paths[("reel3", backend, video_id)] = assemble_reel(video_id, clips, f"reel3_{backend}")
    return paths


# reel_paths = assemble_all_reels(reel1_clips, reel2_clips, reel3_clips)


## 9. Evaluation metrics

All descriptive — no inferential testing, consistent with Stage 2's
treatment of anything beyond its single pre-specified comparison. These
metrics characterise *what kind of reel each strategy produces*, not which
one is "better": phase coverage and compression ratio describe the reel on
its own terms; the two cosine-similarity metrics describe redundancy and
representativeness against the source video's own visual embeddings; the
Spearman correlation asks, for strategies that never saw a saliency score
(Reel 2) or only saw a saliency-filtered subset (Reel 3), how much their
picks agree with saliency anyway; the Jaccard metric compares selections
both across strategies (Reel 1 vs 2 vs 3) and across the three LLM
backends within a strategy.


In [ ]:
def get_manifest_positions(manifest: pd.DataFrame, video_id: str, start_ts: float, end_ts: float) -> np.ndarray:
    vm = manifest[manifest.video_id == video_id].sort_values("frame_idx").reset_index(drop=True)
    mask = (vm.timestamp >= start_ts) & (vm.timestamp <= end_ts)
    return np.flatnonzero(mask.values)


def clip_embedding(manifest: pd.DataFrame, video_id: str, start_ts: float, end_ts: float, backbone_name: str) -> np.ndarray:
    feats = load_cached_features(backbone_name, video_id)
    idx = get_manifest_positions(manifest, video_id, start_ts, end_ts)
    idx = idx[idx < len(feats)]
    return feats[idx].mean(axis=0)


def phase_coverage(clips: pd.DataFrame, manifest: pd.DataFrame, video_id: str) -> float:
    phases_in_video = set(manifest[manifest.video_id == video_id].phase.unique())
    phases_in_reel = set(clips.phase.unique())
    return len(phases_in_reel & phases_in_video) / len(phases_in_video) if phases_in_video else float("nan")


def compression_ratio(clips: pd.DataFrame, manifest: pd.DataFrame, video_id: str) -> dict:
    vm = manifest[manifest.video_id == video_id]
    original_duration = vm.timestamp.max() - vm.timestamp.min()
    raw_duration = clips.duration_s.sum()
    return {
        "raw_compression_ratio": raw_duration / original_duration,
        "final_compression_ratio": (raw_duration / CFG.playback_speed) / original_duration,
    }


def intra_reel_cosine_similarity(clips: pd.DataFrame, manifest: pd.DataFrame, video_id: str, backbone_name: str) -> float:
    if len(clips) < 2:
        return float("nan")
    embeddings = np.stack([
        clip_embedding(manifest, video_id, row.start_ts, row.end_ts, backbone_name)
        for row in clips.itertuples()
    ])
    sims = cosine_similarity(embeddings)
    iu = np.triu_indices(len(sims), k=1)
    return sims[iu].mean()


def reel_to_video_cosine_similarity(clips: pd.DataFrame, manifest: pd.DataFrame, video_id: str, backbone_name: str) -> float:
    reel_embedding = np.stack([
        clip_embedding(manifest, video_id, row.start_ts, row.end_ts, backbone_name)
        for row in clips.itertuples()
    ]).mean(axis=0)
    video_embedding = load_cached_features(backbone_name, video_id).mean(axis=0)
    return cosine_similarity(reel_embedding[None, :], video_embedding[None, :])[0, 0]


def attach_mean_saliency(candidates: pd.DataFrame, saliency: pd.DataFrame, video_id: str) -> pd.DataFrame:
    sal = saliency[saliency.video_id == video_id].sort_values("timestamp")
    out = candidates.copy()
    out["mean_saliency"] = [
        sal[(sal.timestamp >= row.start_ts) & (sal.timestamp <= row.end_ts)].saliency_smooth.mean()
        for row in candidates.itertuples()
    ]
    return out


def spearman_alignment(all_candidates_with_saliency: pd.DataFrame, selected_clip_ids: set) -> float:
    if all_candidates_with_saliency.mean_saliency.nunique() < 2:
        return float("nan")
    selected_flag = all_candidates_with_saliency.clip_id.isin(selected_clip_ids).astype(int)
    rho, _ = spearmanr(all_candidates_with_saliency.mean_saliency, selected_flag)
    return rho


def frame_coverage_mask(clips: pd.DataFrame, manifest: pd.DataFrame, video_id: str) -> np.ndarray:
    n = (manifest.video_id == video_id).sum()
    mask = np.zeros(n, dtype=bool)
    for row in clips.itertuples():
        idx = get_manifest_positions(manifest, video_id, row.start_ts, row.end_ts)
        mask[idx[idx < n]] = True
    return mask


def jaccard_similarity(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    union = (mask_a | mask_b).sum()
    return (mask_a & mask_b).sum() / union if union else float("nan")


def inter_reel_jaccard(video_id: str, reels: dict, manifest: pd.DataFrame) -> pd.DataFrame:
    '''reels: {reel_variant_name: clips_df} for one video, e.g.
    {"reel1": ..., "reel2_chatgpt": ..., "reel3_claude": ...}.'''
    names = list(reels.keys())
    masks = {name: frame_coverage_mask(clips, manifest, video_id) for name, clips in reels.items()}
    matrix = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if j <= i:
                continue
            matrix.loc[a, b] = matrix.loc[b, a] = jaccard_similarity(masks[a], masks[b])
    return matrix


### 9.1 Running all metrics


In [ ]:
def load_own_candidate_pool(video_id: str, reel_variant: str, saliency_candidates_all: pd.DataFrame,
                            saliency: pd.DataFrame) -> pd.DataFrame:
    '''The universe of clips the given reel variant actually chose from,
    each annotated with mean_saliency — used only for the Spearman
    alignment metric, so it must be the *same* candidate pool that
    strategy saw, not some other reel's pool. Reel 1/3 candidates already
    carry mean_saliency (Reel 3's pool is literally Reel 1's saliency
    segments, captioned); Reel 2's sliding-window candidates don't, since
    saliency was never computed for them during selection, so it's
    attached here post hoc for evaluation only.'''
    if reel_variant == "reel1":
        pool = saliency_candidates_all[saliency_candidates_all.video_id == video_id].reset_index(drop=True)
        pool = pool.assign(clip_id=[f"{video_id}_sal_{i}" for i in range(len(pool))])
    elif reel_variant.startswith("reel3_"):
        path = DIRS["candidates"] / f"reel3_candidates_{video_id}.csv"
        pool = pd.read_csv(path) if path.exists() else pd.DataFrame()
    elif reel_variant.startswith("reel2_"):
        path = DIRS["candidates"] / f"reel2_candidates_{video_id}.csv"
        pool = pd.read_csv(path) if path.exists() else pd.DataFrame()
        if len(pool):
            pool = attach_mean_saliency(pool, saliency, video_id)
    else:
        pool = pd.DataFrame()
    return pool


def run_reel_metrics(video_id: str, reel_variant: str, clips: pd.DataFrame, manifest: pd.DataFrame,
                      saliency_candidates_all: pd.DataFrame, saliency: pd.DataFrame) -> dict:
    row = {"video_id": video_id, "reel_variant": reel_variant, "n_clips": len(clips)}
    row["phase_coverage"] = phase_coverage(clips, manifest, video_id)
    row.update(compression_ratio(clips, manifest, video_id))
    row["intra_reel_cosine_similarity"] = intra_reel_cosine_similarity(clips, manifest, video_id, CFG.winning_backbone)
    row["reel_to_video_cosine_similarity"] = reel_to_video_cosine_similarity(clips, manifest, video_id, CFG.winning_backbone)

    pool = load_own_candidate_pool(video_id, reel_variant, saliency_candidates_all, saliency)
    if len(pool):
        selected_spans = set(zip(clips.start_ts.round(1), clips.end_ts.round(1)))
        selected_ids = {
            r.clip_id for r in pool.itertuples()
            if (round(r.start_ts, 1), round(r.end_ts, 1)) in selected_spans
        }
        row["spearman_saliency_alignment"] = spearman_alignment(pool, selected_ids)
    else:
        row["spearman_saliency_alignment"] = float("nan")
    return row


def run_all_metrics(all_reels: dict, manifest: pd.DataFrame, saliency_candidates: pd.DataFrame,
                     saliency: pd.DataFrame) -> pd.DataFrame:
    '''all_reels: {(video_id, reel_variant_name): clips_df}, e.g. keys like
    ("1", "reel1"), ("1", "reel2_chatgpt"), ("1", "reel3_claude").'''
    rows = [
        run_reel_metrics(video_id, variant, clips, manifest, saliency_candidates, saliency)
        for (video_id, variant), clips in all_reels.items()
    ]
    metrics = pd.DataFrame(rows)
    metrics.to_csv(DIRS["metrics"] / "reel_metrics.csv", index=False)
    return metrics


def run_all_jaccard(all_reels: dict, manifest: pd.DataFrame) -> dict:
    matrices = {}
    for video_id in CFG.videos:
        reels_for_video = {variant: clips for (vid, variant), clips in all_reels.items() if vid == video_id}
        if len(reels_for_video) < 2:
            continue
        matrix = inter_reel_jaccard(video_id, reels_for_video, manifest)
        matrix.to_csv(DIRS["metrics"] / f"jaccard_{video_id}.csv")
        matrices[video_id] = matrix
    return matrices


# all_reels = {}
# for video_id, clips in reel1_clips.items():
#     all_reels[(video_id, "reel1")] = clips
# for (backend, video_id), clips in reel2_clips.items():
#     all_reels[(video_id, f"reel2_{backend}")] = clips
# for (backend, video_id), clips in reel3_clips.items():
#     all_reels[(video_id, f"reel3_{backend}")] = clips
# metrics = run_all_metrics(all_reels, manifest, saliency_candidates, saliency)
# jaccard_matrices = run_all_jaccard(all_reels, manifest)


## 10. Outputs

Per-video, per-reel clip selections, candidate pools, and LLM rationales
are already saved to disk as each section runs (Sections 4, 5, 6, 7). This
section compiles the scattered per-`(reel, backend, video)` rationale files
into one supplementary JSON, and writes a run-metadata record (config
snapshot + git hash) matching Stage 2's `save_run_metadata`.


In [ ]:
def compile_all_rationales() -> dict:
    compiled = {}
    for path in sorted(DIRS["selections"].glob("*_rationale.json")):
        with open(path) as f:
            compiled[path.stem.replace("_rationale", "")] = json.load(f)
    out_path = DIRS["results"] / "all_llm_rationales.json"
    with open(out_path, "w") as f:
        json.dump(compiled, f, indent=2)
    print(f"Saved {out_path} ({len(compiled)} reel/backend/video rationale sets)")
    return compiled


def save_run_metadata(extra: dict = None):
    try:
        git_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(ROOT)).decode().strip()
    except Exception:
        git_hash = "unknown (not a git checkout, or git unavailable)"

    meta = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "git_hash": git_hash,
        "config": {
            "videos": CFG.videos,
            "winning_run_name": CFG.winning_run_name,
            "winning_backbone": CFG.winning_backbone,
            "saliency_thresholds": CFG.saliency_thresholds,
            "saliency_operating_threshold": CFG.saliency_operating_threshold,
            "reel_target_min_s": CFG.reel_target_min_s,
            "reel_target_max_s": CFG.reel_target_max_s,
            "reel_leeway_s": CFG.reel_leeway_s,
            "playback_speed": CFG.playback_speed,
            "caption_window_s_list": CFG.caption_window_s_list,
            "caption_window_primary_s": CFG.caption_window_primary_s,
            "llm_backends": CFG.llm_backends,
            "video_crf": CFG.video_crf,
        },
    }
    if extra:
        meta.update(extra)

    out_path = DIRS["results"] / f"run_metadata_{meta['timestamp'].replace(':', '-')}.json"
    with open(out_path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved run metadata: {out_path}")
    return meta


# compile_all_rationales()
# save_run_metadata()


## 11. Putting it together

Uncomment and run in order once `CFG` points at a project root where
Stage 2 has already finished (manifest, feature caches, and — critically —
the LOOCV fold checkpoints for `CFG.winning_run_name`), `CFG.winning_backbone`
is filled in, and at least one LLM backend is configured (`OPENAI_API_KEY`
/ `ANTHROPIC_API_KEY` env vars, or a reachable `CFG.ollama_host`).


In [ ]:
# --- 1. Preprocessing ---
# strip_audio_all_videos()

# --- 2. Load manifest ---
# manifest = load_manifest()

# --- 3. Saliency ---
# saliency_raw = compute_all_frame_saliency()
# saliency = normalize_and_smooth_all(saliency_raw)
# threshold_sweep = sweep_thresholds(saliency)
# saliency_candidates = build_candidate_pool(saliency)  # uses CFG.saliency_operating_threshold

# --- 4. Reel 1 (also fixes the clips-per-phase quota for Reels 2/3) ---
# reel1_clips, clips_per_phase_quota = build_reel1_all(saliency_candidates)

# --- 5. Captioning ---
# frame_captions = build_frame_captions_all(saliency)
# caption_window_comparison = compare_caption_window_sizes(frame_captions)

# --- 6. Reel 2 ---
# reel2_clips = build_reel2_all(frame_captions)

# --- 7. Reel 3 ---
# reel3_clips = build_reel3_all(saliency_candidates, frame_captions)

# --- 8. Assembly ---
# reel_paths = assemble_all_reels(reel1_clips, reel2_clips, reel3_clips)

# --- 9. Metrics ---
# all_reels = {}
# for video_id, clips in reel1_clips.items():
#     all_reels[(video_id, "reel1")] = clips
# for (backend, video_id), clips in reel2_clips.items():
#     all_reels[(video_id, f"reel2_{backend}")] = clips
# for (backend, video_id), clips in reel3_clips.items():
#     all_reels[(video_id, f"reel3_{backend}")] = clips
# metrics = run_all_metrics(all_reels, manifest, saliency_candidates, saliency)
# jaccard_matrices = run_all_jaccard(all_reels, manifest)

# --- 10. Outputs ---
# compile_all_rationales()
# save_run_metadata()
